# Chapter 4: Database Updater Module

This chapter covers database update workflows:
- Data validation
- Incremental updates
- Batch processing
- Error handling and recovery

## 1. Setup and Imports

In [ ]:
import datetime
import os
import sys
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Any

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sqlalchemy import text

# Add project root to path
project_root = os.path.dirname(os.path.abspath('__file__'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ETF_DB, DATA_DIR, setup_logging, OUTPUT_DIR
from utils import DataLoader

logger = setup_logging('database_updater')

## 2. Configuration Classes

In [ ]:
@dataclass
class ValidationConfig:
    """Data validation configuration"""
    required_fields: List[str]
    field_types: Dict[str, type]
    value_ranges: Dict[str, tuple] = None

@dataclass
class UpdateResult:
    """Update operation result"""
    success: bool
    rows_affected: int = 0
    error: str = None
    details: str = None

## 3. Base Updater Class

In [ ]:
class BaseUpdater:
    """Base class for data updaters"""
    
    def __init__(self, table_name: str, key_fields: List[str], 
                 data_loader: DataLoader = None):
        """Initialize base updater
        
        Args:
            table_name: Target table name
            key_fields: Primary key fields
            data_loader: DataLoader instance
        """
        self.table_name = table_name
        self.key_fields = key_fields
        self.data_loader = data_loader or DataLoader()
        self.logger = setup_logging(f'updater.{table_name}')
    
    def validate_data(self, df: pd.DataFrame, 
                      config: ValidationConfig) -> List[str]:
        """Validate DataFrame before update
        
        Args:
            df: DataFrame to validate
            config: Validation configuration
            
        Returns:
            List of validation errors
        """
        errors = []
        
        # Check required fields
        for field in config.required_fields:
            if field not in df.columns:
                errors.append(f"Missing required field: {field}")
            elif df[field].isna().any():
                errors.append(f"Field {field} contains null values")
        
        # Check field types
        for field, expected_type in config.field_types.items():
            if field in df.columns:
                try:
                    df[field].astype(expected_type)
                except (ValueError, TypeError) as e:
                    errors.append(f"Field {field} type error: {str(e)}")
        
        # Check value ranges
        if config.value_ranges:
            for field, (min_val, max_val) in config.value_ranges.items():
                if field in df.columns:
                    if min_val is not None and df[field].min() < min_val:
                        errors.append(f"Field {field} has values below minimum {min_val}")
                    if max_val is not None and df[field].max() > max_val:
                        errors.append(f"Field {field} has values above maximum {max_val}")
        
        return errors
    
    def backup_data(self, condition: str = None) -> pd.DataFrame:
        """Backup existing data
        
        Args:
            condition: WHERE clause for backup selection
            
        Returns:
            DataFrame with backed up data
        """
        sql = f"SELECT * FROM {self.table_name}"
        if condition:
            sql += f" WHERE {condition}"
        
        backup_df = self.data_loader.query_df(sql)
        
        # Save to backup file
        backup_dir = DATA_DIR / 'backup'
        backup_dir.mkdir(parents=True, exist_ok=True)
        
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_file = backup_dir / f"{self.table_name}_{timestamp}.csv"
        backup_df.to_csv(backup_file, index=False, encoding='utf-8-sig')
        
        self.logger.info(f"Backup saved to {backup_file}")
        return backup_df
    
    def upsert_data(self, df: pd.DataFrame) -> UpdateResult:
        """Upsert data to table (insert or update)
        
        Args:
            df: DataFrame to upsert
            
        Returns:
            UpdateResult object
        """
        try:
            # Delete existing records
            if df.empty:
                return UpdateResult(success=True, rows_affected=0)
            
            # Build delete condition
            key_values = df[self.key_fields].drop_duplicates()
            
            with self.data_loader.engine.begin() as conn:
                for _, row in key_values.iterrows():
                    conditions = []
                    for field in self.key_fields:
                        conditions.append(f"{field} = '{row[field]}'")
                    
                    delete_sql = f"DELETE FROM {self.table_name} WHERE {' AND '.join(conditions)}"
                    conn.execute(text(delete_sql))
                
                # Insert new records
                rows_affected = df.to_sql(
                    name=self.table_name.split('.')[-1],
                    con=conn,
                    if_exists='append',
                    index=False
                )
            
            return UpdateResult(
                success=True,
                rows_affected=len(df),
                details=f"Updated {len(df)} records"
            )
            
        except Exception as e:
            self.logger.error(f"Upsert failed: {str(e)}")
            return UpdateResult(
                success=False,
                error=str(e)
            )

# Initialize base updater
base_updater = BaseUpdater(
    table_name='fund.marketinfo',
    key_fields=['TRADE_CODE', 'DT']
)
print("Base updater initialized")

## 4. ETF Data Updater

In [ ]:
class ETFUpdater(BaseUpdater):
    """ETF specific data updater"""
    
    def __init__(self, data_loader: DataLoader = None):
        """Initialize ETF updater
        
        Args:
            data_loader: DataLoader instance
        """
        super().__init__(
            table_name='fund.marketinfo',
            key_fields=['TRADE_CODE', 'DT'],
            data_loader=data_loader
        )
    
    def get_missing_dates(self, start_date: str, end_date: str) -> List[str]:
        """Find dates with missing data
        
        Args:
            start_date: Start date
            end_date: End date
            
        Returns:
            List of missing dates
        """
        # Get trading days
        trading_days_sql = """
        SELECT DISTINCT DT FROM fund.marketinfo
        WHERE DT BETWEEN :start AND :end
        ORDER BY DT
        """
        trading_days = self.data_loader.query_df(
            trading_days_sql, (start_date, end_date)
        )['DT'].tolist()
        
        # Get ETF list
        etf_list_sql = """
        SELECT TRADE_CODE FROM yq.etf_base_info
        WHERE DT = (SELECT MAX(DT) FROM yq.etf_base_info)
        """
        etf_list = self.data_loader.query_df(etf_list_sql)['TRADE_CODE'].tolist()
        
        # Check for missing data
        missing_dates = []
        for date in trading_days:
            check_sql = f"""
            SELECT COUNT(DISTINCT TRADE_CODE) as cnt
            FROM fund.marketinfo
            WHERE DT = '{date}'
            """
            count = self.data_loader.query_df(check_sql)['cnt'].iloc[0]
            
            if count < len(etf_list) * 0.9:  # Less than 90% coverage
                missing_dates.append(date.strftime('%Y-%m-%d'))
        
        return missing_dates
    
    def update_nav_data(self, nav_data: pd.DataFrame) -> UpdateResult:
        """Update NAV data
        
        Args:
            nav_data: DataFrame with NAV data
            
        Returns:
            UpdateResult object
        """
        # Validate data
        validation_config = ValidationConfig(
            required_fields=['TRADE_CODE', 'DT', 'NAV', 'NAV_ADJ'],
            field_types={
                'TRADE_CODE': str,
                'NAV': float,
                'NAV_ADJ': float
            },
            value_ranges={
                'NAV': (0, None),
                'NAV_ADJ': (0, None)
            }
        )
        
        errors = self.validate_data(nav_data, validation_config)
        if errors:
            return UpdateResult(
                success=False,
                error=f"Validation errors: {'; '.join(errors)}"
            )
        
        return self.upsert_data(nav_data)
    
    def update_unit_data(self, unit_data: pd.DataFrame) -> UpdateResult:
        """Update unit/share data
        
        Args:
            unit_data: DataFrame with unit data
            
        Returns:
            UpdateResult object
        """
        # Update table name for unit data
        self.table_name = 'fund.marketinfo_unit'
        
        validation_config = ValidationConfig(
            required_fields=['TRADE_CODE', 'DT', 'UNIT_TOTAL'],
            field_types={
                'TRADE_CODE': str,
                'UNIT_TOTAL': float
            },
            value_ranges={
                'UNIT_TOTAL': (0, None)
            }
        )
        
        errors = self.validate_data(unit_data, validation_config)
        if errors:
            return UpdateResult(
                success=False,
                error=f"Validation errors: {'; '.join(errors)}"
            )
        
        return self.upsert_data(unit_data)

# Initialize ETF updater
etf_updater = ETFUpdater()
print("ETF updater initialized")

## 5. Batch Update Process

In [ ]:
class BatchUpdateManager:
    """Manager for batch update operations"""
    
    def __init__(self, batch_size: int = 1000):
        """Initialize batch manager
        
        Args:
            batch_size: Number of records per batch
        """
        self.batch_size = batch_size
        self.logger = setup_logging('batch_updater')
    
    def process_batches(self, df: pd.DataFrame, 
                        update_func, **kwargs) -> Dict[str, Any]:
        """Process DataFrame in batches
        
        Args:
            df: DataFrame to process
            update_func: Update function to call
            **kwargs: Additional arguments for update_func
            
        Returns:
            Summary dictionary
        """
        total_rows = len(df)
        total_batches = (total_rows + self.batch_size - 1) // self.batch_size
        
        results = {
            'total_rows': total_rows,
            'total_batches': total_batches,
            'successful_batches': 0,
            'failed_batches': 0,
            'total_rows_updated': 0,
            'errors': []
        }
        
        for i in range(0, total_rows, self.batch_size):
            batch_num = i // self.batch_size + 1
            batch_df = df.iloc[i:i + self.batch_size]
            
            self.logger.info(f"Processing batch {batch_num}/{total_batches}")
            
            try:
                result = update_func(batch_df, **kwargs)
                
                if result.success:
                    results['successful_batches'] += 1
                    results['total_rows_updated'] += result.rows_affected
                else:
                    results['failed_batches'] += 1
                    results['errors'].append({
                        'batch': batch_num,
                        'error': result.error
                    })
                    
            except Exception as e:
                results['failed_batches'] += 1
                results['errors'].append({
                    'batch': batch_num,
                    'error': str(e)
                })
        
        # Print summary
        print("\nBatch Update Summary")
        print("=" * 50)
        print(f"Total rows: {results['total_rows']}")
        print(f"Total batches: {results['total_batches']}")
        print(f"Successful batches: {results['successful_batches']}")
        print(f"Failed batches: {results['failed_batches']}")
        print(f"Total rows updated: {results['total_rows_updated']}")
        
        if results['errors']:
            print(f"\nErrors:")
            for err in results['errors']:
                print(f"  Batch {err['batch']}: {err['error']}")
        
        return results

# Initialize batch manager
batch_manager = BatchUpdateManager(batch_size=500)
print("Batch manager initialized")

## 6. Data Sync Utilities

In [ ]:
def sync_etf_data(start_date: str = None, end_date: str = None,
                  dry_run: bool = True) -> Dict[str, Any]:
    """Sync ETF data between tables
    
    Args:
        start_date: Start date for sync
        end_date: End date for sync
        dry_run: If True, only report what would be synced
        
    Returns:
        Sync summary dictionary
    """
    loader = DataLoader()
    
    # Get date range
    if not start_date:
        start_date = (datetime.datetime.now() - datetime.timedelta(days=30)).strftime('%Y-%m-%d')
    if not end_date:
        end_date = datetime.datetime.now().strftime('%Y-%m-%d')
    
    print(f"Syncing ETF data from {start_date} to {end_date}")
    print(f"Dry run: {dry_run}")
    
    # Get current data status
    nav_count_sql = f"""
    SELECT COUNT(*) as cnt FROM fund.marketinfo
    WHERE DT BETWEEN '{start_date}' AND '{end_date}'
    """
    nav_count = loader.query_df(nav_count_sql)['cnt'].iloc[0]
    
    unit_count_sql = f"""
    SELECT COUNT(*) as cnt FROM fund.marketinfo_unit
    WHERE DT BETWEEN '{start_date}' AND '{end_date}'
    """
    unit_count = loader.query_df(unit_count_sql)['cnt'].iloc[0]
    
    summary = {
        'start_date': start_date,
        'end_date': end_date,
        'dry_run': dry_run,
        'nav_records': nav_count,
        'unit_records': unit_count
    }
    
    print(f"\nCurrent data status:")
    print(f"  NAV records: {nav_count}")
    print(f"  Unit records: {unit_count}")
    
    return summary

# Run sync check
sync_summary = sync_etf_data(dry_run=True)

## 7. Update Log and Monitoring

In [ ]:
def get_update_status(table_name: str = 'fund.marketinfo') -> pd.DataFrame:
    """Get data update status
    
    Args:
        table_name: Table to check
        
    Returns:
        DataFrame with update status
    """
    loader = DataLoader()
    
    sql = f"""
    SELECT 
        DT as date,
        COUNT(*) as record_count,
        COUNT(DISTINCT TRADE_CODE) as unique_codes
    FROM {table_name}
    WHERE DT >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
    GROUP BY DT
    ORDER BY DT DESC
    """
    
    status_df = loader.query_df(sql)
    
    print(f"Update Status for {table_name} (Last 30 Days):")
    print(status_df.to_string(index=False))
    
    return status_df

# Get update status
# status = get_update_status()

## 8. Error Recovery

In [ ]:
def recover_from_backup(backup_file: str, table_name: str) -> UpdateResult:
    """Recover data from backup file
    
    Args:
        backup_file: Path to backup CSV file
        table_name: Target table name
        
    Returns:
        UpdateResult object
    """
    try:
        # Load backup data
        backup_df = pd.read_csv(backup_file)
        
        if backup_df.empty:
            return UpdateResult(
                success=False,
                error="Backup file is empty"
            )
        
        # Create updater and restore
        updater = BaseUpdater(table_name, ['TRADE_CODE', 'DT'])
        result = updater.upsert_data(backup_df)
        
        return result
        
    except Exception as e:
        return UpdateResult(
            success=False,
            error=f"Recovery failed: {str(e)}"
        )

print("Recovery function defined")

## 9. Summary

This chapter covered:
1. Base updater class with validation and backup
2. ETF-specific data updater
3. Batch update processing
4. Data synchronization utilities
5. Update status monitoring
6. Error recovery from backup

### Key Classes
- `BaseUpdater`: Base class for all updaters
- `ETFUpdater`: ETF-specific update logic
- `BatchUpdateManager`: Batch processing manager

### Best Practices
- Always validate data before updates
- Create backups before major changes
- Use batch processing for large datasets
- Monitor update status regularly
- Implement error recovery procedures